In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("../data/processed/wine_processed.csv")

X = df.drop("quality", axis=1)
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV Score:", rf_grid.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}
Best CV Score: 0.6104976141785958


In [5]:
rf_best = rf_grid.best_estimator_

rf_pred = rf_best.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.5882352941176471


In [6]:
svc_params = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

svc_grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid=svc_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

svc_grid.fit(X_train_scaled, y_train)

print("Best Parameters:", svc_grid.best_params_)
print("Best CV Score:", svc_grid.best_score_)

Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV Score: 0.6031356509884116


c:\Users\Diya\OneDrive\Desktop\ML\Wine Quality Prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [7]:
svc_best = svc_grid.best_estimator_

svc_pred = svc_best.predict(X_test_scaled)

svc_accuracy = accuracy_score(y_test, svc_pred)

print("SVC Accuracy:", svc_accuracy)

SVC Accuracy: 0.6029411764705882


In [8]:
sgd_params = {
    "loss": ["hinge", "log_loss", "modified_huber"],
    "penalty": ["l2", "l1", "elasticnet"],
    "alpha": [0.0001, 0.001, 0.01]
}

sgd_random = RandomizedSearchCV(
    SGDClassifier(random_state=42),
    param_distributions=sgd_params,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

sgd_random.fit(X_train_scaled, y_train)

print("Best Parameters:", sgd_random.best_params_)
print("Best CV Score:", sgd_random.best_score_)

Best Parameters: {'penalty': 'l2', 'loss': 'log_loss', 'alpha': 0.01}
Best CV Score: 0.5834886010755131


In [9]:
sgd_best = sgd_random.best_estimator_

sgd_pred = sgd_best.predict(X_test_scaled)

sgd_accuracy = accuracy_score(y_test, sgd_pred)

print("SGD Accuracy:", sgd_accuracy)

SGD Accuracy: 0.5833333333333334


In [10]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "SVC",
        "SGD"
    ],

    "Accuracy": [
        rf_accuracy,
        svc_accuracy,
        sgd_accuracy
    ]
})

comparison.sort_values(
    by="Accuracy",
    ascending=False,
    inplace=True
)

comparison

,Model,Accuracy
1,SVC,0.602941
0,Random Forest,0.588235
2,SGD,0.583333


In [12]:
import os
import joblib

# Create artifacts folder if it doesn't exist
os.makedirs("../artifacts", exist_ok=True)

# Save the best SVC model
joblib.dump(
    svc_grid.best_estimator_,
    "../artifacts/best_model.pkl"
)

# Save the scaler
joblib.dump(
    scaler,
    "../artifacts/scaler.pkl"
)

print("✅ Best model and scaler saved successfully!")

✅ Best model and scaler saved successfully!


In [13]:
import joblib

model = joblib.load("../artifacts/best_model.pkl")

print(model)
print("Probability Enabled:", model.probability)

SVC(C=1, probability=True, random_state=42)
Probability Enabled: True
